# Data Cleaning

In [1]:
%reset -f

Imports

In [2]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import gc

Loading the datasheets

In [3]:
DATA_RAW = Path("../data/raw")

studentInfo = pd.read_csv(DATA_RAW / "studentInfo.csv")
studentVle = pd.read_csv(DATA_RAW / "studentVle.csv")
assessments = pd.read_csv(DATA_RAW / "assessments.csv")
studentAssessment = pd.read_csv(DATA_RAW / "studentAssessment.csv")
courses = pd.read_csv(DATA_RAW / "courses.csv")
vle = pd.read_csv(DATA_RAW / "vle.csv")
studentReg = pd.read_csv(DATA_RAW / "studentRegistration.csv")

### Tables Overview

In [4]:
datasets = {
    "studentInfo": studentInfo,
    "studentVle": studentVle,
    "assessments": assessments,
    "studentAssessment": studentAssessment,
    "courses": courses,
    "vle": vle,
    "studentRegistration": studentReg
}

for name, df in datasets.items():
    print(f"\n{'='*40}")
    print(f"{name}")
    print(f"{'='*40}")
    print("Shape:", df.shape)
    print("\nMissing values:\n", df.isnull().sum())
    print("\nDuplicate rows:", df.duplicated().sum())


studentInfo
Shape: (32593, 12)

Missing values:
 code_module                0
code_presentation          0
id_student                 0
gender                     0
region                     0
highest_education          0
imd_band                1111
age_band                   0
num_of_prev_attempts       0
studied_credits            0
disability                 0
final_result               0
dtype: int64

Duplicate rows: 0

studentVle
Shape: (10655280, 6)

Missing values:
 code_module          0
code_presentation    0
id_student           0
id_site              0
date                 0
sum_click            0
dtype: int64

Duplicate rows: 787170

assessments
Shape: (206, 6)

Missing values:
 code_module           0
code_presentation     0
id_assessment         0
assessment_type       0
date                 11
weight                0
dtype: int64

Duplicate rows: 0

studentAssessment
Shape: (173912, 5)

Missing values:
 id_assessment       0
id_student          0
date_submitted      0

Handling Missing Values

In [5]:
studentAssessment["date_submitted"] = pd.to_numeric(studentAssessment["date_submitted"], errors="coerce")
studentAssessment["date_submitted"] = studentAssessment["date_submitted"].clip(lower=0)  # keeps NaN as NaN
studentAssessment["submitted_flag"] = studentAssessment["date_submitted"].notna().astype(int)
studentAssessment["score_missing_flag"] = studentAssessment["score"].isna().astype(int)

Cleaned studentRegistration 
Removed invalid registrations where date\_unregistration exists and is <= 0 \(unregistered before/at course start\)\.
Added derived flags:
registration\_missing\_flag \(if date\_registration is missing\)
unregistered\_flag \(if date\_unregistration exists\)
unregistration\_week \(unregistration date converted to week; missing → \-1\)
Filled missing date\_registration with 0
late\_registration\_flag \(registered after day 0\)

In [6]:
studentReg["date_unregistration"] = pd.to_numeric(studentReg["date_unregistration"], errors="coerce")
studentReg["date_registration"] = pd.to_numeric(studentReg["date_registration"], errors="coerce")

# Remove students who unregistered BEFORE course start (negative unregistration dates)
studentReg = studentReg[~(studentReg["date_unregistration"].notna() & (studentReg["date_unregistration"] < 0))].copy()

studentReg["registration_missing_flag"] = studentReg["date_registration"].isna().astype(int)
studentReg["unregistered_flag"] = studentReg["date_unregistration"].notna().astype(int)
studentReg["unregistration_week"] = (studentReg["date_unregistration"] // 7).fillna(-1).astype(int)

# Fill missing registration with 0 (treat as registered at course start)
studentReg["date_registration"] = studentReg["date_registration"].fillna(0)
studentReg["late_registration_flag"] = (studentReg["date_registration"] > 0).astype(int)

The Courses table and Student Registration table were checked for invalid registration or invalid courses, and the dataset was found to be consistent with no invalid data and no duplicates\.

In [7]:
courses = courses.copy()
# ensuring numeric values for the presentation length and converting them to no of weeks.
courses["module_presentation_length"] = pd.to_numeric(courses["module_presentation_length"],errors="coerce")
courses["course_weeks"] = np.ceil(courses["module_presentation_length"] / 7).astype("Int64")

# Validation
print("Unique course week lengths:")
print(courses["course_weeks"].value_counts(dropna=False))

non_stem_modules = {"AAA", "BBB", "GGG"}
# creating course_type_stem column to identify STEM/non-STEM 0=Non-STEM, 1=STEM
courses["course_type_stem"] = (~courses["code_module"].astype(str).str.upper().str.strip().isin(non_stem_modules)).astype(int) 

display(courses.head())

Unique course week lengths:
course_weeks
39      9
35      8
38      4
34      1
<NA>    0
Name: count, dtype: Int64


,code_module,code_presentation,module_presentation_length,course_weeks,course_type_stem
0,AAA,2013J,268,39,0
1,AAA,2014J,269,39,0
2,BBB,2013J,268,39,0
3,BBB,2014J,262,38,0
4,BBB,2013B,240,35,0


In [8]:
KEYS = ["id_student", "code_module", "code_presentation"]
COURSE_KEYS = ["code_module", "code_presentation"]

# windows per course
course_windows = courses[COURSE_KEYS + ["course_weeks"]].drop_duplicates().copy()
# course_windows["course_weeks"] = pd.to_numeric(course_windows["course_weeks"], errors="coerce")
course_windows["course_weeks"] = course_windows["course_weeks"].fillna(1).clip(lower=1).astype(int)

course_windows["q1_end"] = np.ceil(course_windows["course_weeks"] * 0.25).astype(int)
course_windows["q2_end"] = np.ceil(course_windows["course_weeks"] * 0.50).astype(int)
course_windows["q3_end"] = np.ceil(course_windows["course_weeks"] * 0.75).astype(int)
course_windows["q4_end"] = course_windows["course_weeks"].astype(int)

course_windows["q1_end"] = course_windows["q1_end"].clip(lower=1)
course_windows["q2_end"] = course_windows[["q2_end", "course_weeks"]].min(axis=1)
course_windows["q3_end"] = course_windows[["q3_end", "course_weeks"]].min(axis=1)
course_windows["q4_end"] = course_windows["q4_end"].clip(lower=1)

Demographics cleaning/encoding

In [9]:
studentInfo = studentInfo.copy()

# Pass/Distinction = 0, Fail/Withdrawn = 1
studentInfo["target_at_risk"] = (studentInfo["final_result"].isin(["Fail", "Withdrawn"]).astype(int))

#binary encodings for gender and disability
studentInfo["gender_bin"] = (studentInfo["gender"].astype(str).str.upper() == "M").astype(int)
studentInfo["disability_bin"] = (studentInfo["disability"].astype(str).str.upper() == "Y").astype(int)

# ordinal encoding for age group and missing age flag
age_map = {
    "0-35": 0,
    "35-55": 1,
    "55<=": 2,
    "55+": 2
}
studentInfo["age_band_ord"] = studentInfo["age_band"].map(age_map)
studentInfo["age_band_missing_flag"] = studentInfo["age_band"].isna().astype(int)

# highest education ordinal encoding and missing flag
edu_map = {
    "No Formal quals": 0,
    "Lower Than A Level": 1,
    "A Level or Equivalent": 2,
    "HE Qualification": 3,
    "Post Graduate Qualification": 4
}
studentInfo["highest_education_ord"] = studentInfo["highest_education"].map(edu_map)
studentInfo["highest_education_missing_flag"] = studentInfo["highest_education"].isna().astype(int)

# Region - ohe as per paper
# region_ohe = pd.get_dummies(
#     studentInfo_clean["region"],
#     prefix="region",
#     dummy_na=True
# )
# studentInfo_clean = pd.concat([studentInfo_clean, region_ohe], axis=1)

# nation level ohe as request
def map_region_to_nation(region):
    if pd.isna(region):
        return "Unknown"

    r = str(region).lower()

    if "scotland" in r:
        return "Scotland"
    if "wales" in r:
        return "Wales"
    if "ireland" in r:
        return "Northern_Ireland"
    if "england" in r or "region" in r:
        return "England"

    return "Other"

studentInfo["nation"] = studentInfo["region"].apply(map_region_to_nation)

nation_ohe = pd.get_dummies(studentInfo["nation"],prefix="nation",dummy_na=False)
studentInfo = pd.concat([studentInfo, nation_ohe], axis=1)

# imd-band - categorical not ordinal as paper
studentInfo["imd_band"] = studentInfo["imd_band"].fillna("Unknown")

studentInfo["imd_band"] = studentInfo["imd_band"].astype(str).fillna("Unknown")

def bin_imd_5(x: str) -> str:
    x = str(x).strip()
    if x in {"0-10%", "10-20%"}:
        return "VeryLow"
    if x in {"20-30%", "30-40%"}:
        return "Low"
    if x in {"40-50%", "50-60%"}:
        return "Medium"
    if x in {"60-70%", "70-80%"}:
        return "High"
    if x in {"80-90%", "90-100%"}:
        return "VeryHigh"
    return "Unknown"

# 1) 5-category IMD
studentInfo["imd_5cat"] = studentInfo["imd_band"].apply(bin_imd_5)

# 2) Ordinal encoding (recommended)
imd_ord_map = {
    "VeryLow": 0,
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "VeryHigh": 4,
    "Unknown": -1
}
studentInfo["imd_ordinal"] = studentInfo["imd_5cat"].map(imd_ord_map).astype(int)

# drop the original columns for the encoded values
studentInfo.drop(columns=["gender", "disability", "age_band","highest_education", "region", "nation","final_result","imd_band"],inplace=True)

display(studentInfo.head())


,code_module,code_presentation,id_student,num_of_prev_attempts,studied_credits,target_at_risk,gender_bin,disability_bin,age_band_ord,age_band_missing_flag,highest_education_ord,highest_education_missing_flag,nation_England,nation_Northern_Ireland,nation_Scotland,nation_Wales,imd_5cat,imd_ordinal
0,AAA,2013J,11391,0,240,0,1,0,2,0,3,0,True,False,False,False,VeryHigh,4
1,AAA,2013J,28400,0,60,0,0,0,1,0,3,0,False,False,True,False,Low,1
2,AAA,2013J,30268,0,60,1,0,1,1,0,2,0,True,False,False,False,Low,1
3,AAA,2013J,31604,0,60,0,0,0,1,0,2,0,True,False,False,False,Medium,2
4,AAA,2013J,32885,0,60,0,0,0,0,0,1,0,True,False,False,False,Medium,2


In [10]:
# ---------------------------
# 4) Base static table (this is what you will attach to BOTH master and quarter datasets)
# ---------------------------
# base_static = (
#     studentInfo
#     .merge(studentReg, on=KEYS, how="inner")
#     .merge(courses, on=COURSE_KEYS, how="inner")
#     .merge(course_windows[COURSE_KEYS + ["course_weeks", "q1_end", "q2_end", "q3_end", "q4_end"]], on=COURSE_KEYS, how="inner")
# )
courses_no_weeks = courses.drop(columns=["course_weeks"], errors="ignore")

base_static = (
    studentInfo
    .merge(studentReg, on=KEYS, how="inner")
    .merge(courses_no_weeks, on=COURSE_KEYS, how="inner")
    .merge(course_windows[COURSE_KEYS + ["course_weeks", "q1_end", "q2_end", "q3_end", "q4_end"]],
           on=COURSE_KEYS, how="inner")
)


In [11]:
base_static.head()

,code_module,code_presentation,id_student,num_of_prev_attempts,studied_credits,target_at_risk,gender_bin,disability_bin,age_band_ord,age_band_missing_flag,...,unregistered_flag,unregistration_week,late_registration_flag,module_presentation_length,course_type_stem,course_weeks,q1_end,q2_end,q3_end,q4_end
0,AAA,2013J,11391,0,240,0,1,0,2,0,...,0,-1,0,268,0,39,10,20,30,39
1,AAA,2013J,28400,0,60,0,0,0,1,0,...,0,-1,0,268,0,39,10,20,30,39
2,AAA,2013J,30268,0,60,1,0,1,1,0,...,1,1,0,268,0,39,10,20,30,39
3,AAA,2013J,31604,0,60,0,0,0,1,0,...,0,-1,0,268,0,39,10,20,30,39
4,AAA,2013J,32885,0,60,0,0,0,0,0,...,0,-1,0,268,0,39,10,20,30,39


Aggregating the studentVle sum\_click per day and merging the activity\_type to the student\_vle\_daily

In [12]:
# VLE PIPELINE: Raw -> Daily -> Enriched -> Weekly -> delete unused dfs

studentVle["date"] = pd.to_numeric(studentVle["date"], errors="coerce")
studentVle["sum_click"] = pd.to_numeric(studentVle["sum_click"], errors="coerce").fillna(0)

#Aggregate studentVle to DAILY level
studentVle_daily = (
    studentVle
    .groupby(["id_student", "code_module", "code_presentation", "id_site", "date"], as_index=False)
    .agg(sum_click=("sum_click", "sum"))
)
print("studentVle_daily shape:", studentVle_daily.shape)

#Attach activity_type (lookup)
vle_lookup = vle[["id_site", "activity_type"]].drop_duplicates()

studentVle_daily_enriched = studentVle_daily.merge(
    vle_lookup,
    on="id_site",
    how="left",
    validate="m:1"
)
print("studentVle_daily_enriched shape (before date>=0):", studentVle_daily_enriched.shape)

#Free space
del studentVle_daily
del vle_lookup
del studentVle
gc.collect()

#course starts at day 0
studentVle_daily_enriched = studentVle_daily_enriched[studentVle_daily_enriched["date"] >= 0].copy()

#Week number
studentVle_daily_enriched["week"] = (studentVle_daily_enriched["date"] // 7) + 1

#Map activity_type -> engagement_dim
CONTENT_TYPES = {
    "subpage", "homepage", "oucontent", "resource", "url", "page",
    "folder", "glossary", "htmlactivity", "dualpane", "repeatactivity"
}
ASSESSMENT_TYPES = {"quiz", "externalquiz", "questionnaire"}
SOCIAL_TYPES = {"forumng", "ouwiki", "oucollaborate", "ouelluminate", "dataplus", "sharedsubpage"}

def map_engagement_dim(activity_type):
    if pd.isna(activity_type):
        return "unknown"
    a = str(activity_type).strip().lower()
    if a in CONTENT_TYPES:
        return "content"
    if a in ASSESSMENT_TYPES:
        return "assessment"
    if a in SOCIAL_TYPES:
        return "social"
    return "other"

studentVle_daily_enriched["engagement_dim"] = studentVle_daily_enriched["activity_type"].apply(map_engagement_dim)

#Keep only columns needed for weekly aggregation + quarter features
studentVle_daily_enriched = studentVle_daily_enriched[
    ["id_student", "code_module", "code_presentation", "week", "sum_click", "engagement_dim"]
].copy()

print("studentVle_daily_enriched shape (trimmed):", studentVle_daily_enriched.shape)
display(studentVle_daily_enriched.head())

# 5) Aggregate to WEEKLY (much smaller than daily)
studentVle_weekly = (
    studentVle_daily_enriched
    .groupby(["id_student", "code_module", "code_presentation", "week", "engagement_dim"], as_index=False)
    .agg(sum_click=("sum_click", "sum"))
)
print("studentVle_weekly shape:", studentVle_weekly.shape)

#Free memory
del studentVle_daily_enriched
gc.collect()

studentVle_daily shape: (8459320, 6)
studentVle_daily_enriched shape (before date>=0): (8459320, 7)
studentVle_daily_enriched shape (trimmed): (7859079, 6)


,id_student,code_module,code_presentation,week,sum_click,engagement_dim
0,6516,AAA,2014J,30,6,social
1,6516,AAA,2014J,30,8,social
2,6516,AAA,2014J,32,9,social
3,6516,AAA,2014J,1,38,social
4,6516,AAA,2014J,1,12,social


studentVle_weekly shape: (1085419, 6)


0

In [13]:
# # weekly wide (wkXX_content / wkXX_assessment / wkXX_social)
# dim_keep = {"content", "assessment", "social"}
# weekly_dim_long = (
#     studentVle_weekly[studentVle_weekly["engagement_dim"].isin(dim_keep)]
#     .groupby(KEYS + ["week", "engagement_dim"], as_index=False)
#     .agg(clicks=("sum_click", "sum"))
# )

# weekly_dim_wide = (
#     weekly_dim_long
#     .pivot_table(index=KEYS, columns=["week", "engagement_dim"], values="clicks", fill_value=0.0)
#     .reset_index()
# )

# def _col_to_name(col):
#     if isinstance(col, str):
#         return col
#     if isinstance(col, tuple):
#         parts = [p for p in col if p is not None and p != ""]
#         wk = None
#         dim = None
#         for p in parts:
#             if isinstance(p, (int, np.integer, float, np.floating)) and wk is None:
#                 wk = int(p)
#         str_parts = [str(p) for p in parts if isinstance(p, str)]
#         str_parts = [s for s in str_parts if s.lower() not in {"clicks"}]
#         if len(str_parts) > 0:
#             dim = str_parts[-1]
#         if wk is not None and dim is not None:
#             return f"wk{wk:02d}_{dim}"
#         return "_".join([str(p) for p in parts])
#     return str(col)

# weekly_dim_wide.columns = [_col_to_name(c) for c in weekly_dim_wide.columns]

# # ---------------------------
# # 6) MASTER weekly table = base_static + weekly wide
# # ---------------------------
# master_table = base_static.merge(weekly_dim_wide, on=KEYS, how="left")

# weekly_cols = [c for c in weekly_dim_wide.columns if c not in KEYS]
# master_table[weekly_cols] = master_table[weekly_cols].fillna(0.0)
# eps = 1e-9
# dim_keep = {"content", "assessment", "social"}

# # 1) Keep only the 3 dims
# vle_w = studentVle_weekly[studentVle_weekly["engagement_dim"].isin(dim_keep)].copy()

# # 2) Attach course length and DROP weeks beyond course length
# vle_w = vle_w.merge(
#     course_windows[COURSE_KEYS + ["course_weeks"]],
#     on=COURSE_KEYS,
#     how="inner",
#     validate="m:1"
# )

# vle_w = vle_w[
#     (vle_w["week"] >= 1) &
#     (vle_w["week"] <= vle_w["course_weeks"])
# ].copy()

# # 3) Course totals per (course, week, dim)
# course_week_dim = (
#     vle_w.groupby(COURSE_KEYS + ["week", "engagement_dim"], as_index=False)
#          .agg(course_clicks=("sum_click", "sum"))
# )

# # 4) Normalize
# vle_w = vle_w.merge(
#     course_week_dim,
#     on=COURSE_KEYS + ["week", "engagement_dim"],
#     how="left",
#     validate="m:1"
# )

# vle_w["share_clicks"] = np.where(
#     vle_w["course_clicks"] > 0,
#     vle_w["sum_click"] / (vle_w["course_clicks"] + eps),
#     0.0
# )

# # 5) Pivot to wide
# weekly_share_wide = (
#     vle_w.pivot_table(
#         index=KEYS,
#         columns=["week", "engagement_dim"],
#         values="share_clicks",
#         fill_value=0.0
#     )
#     .reset_index()
# )

# def _col_to_name_norm(col):
#     if isinstance(col, str):
#         return col
#     if isinstance(col, tuple):
#         wk, dim = None, None
#         for p in col:
#             if isinstance(p, (int, np.integer, float, np.floating)) and wk is None:
#                 wk = int(p)
#             if isinstance(p, str):
#                 dim = p
#         if wk is not None and dim is not None:
#             return f"wk{wk:02d}_{dim}_norm"
#     return str(col)

# weekly_share_wide.columns = [_col_to_name_norm(c) for c in weekly_share_wide.columns]

# # 6) Merge into master
# master_table = base_static.merge(weekly_share_wide, on=KEYS, how="left")

# norm_weekly_cols = [c for c in weekly_share_wide.columns if c not in KEYS]
# master_table[norm_weekly_cols] = master_table[norm_weekly_cols].fillna(0.0)

eps = 1e-9
dim_keep = {"content", "assessment", "social"}

# Freeze keys (avoid notebook state issues)
KEYS = ["id_student", "code_module", "code_presentation"]
COURSE_KEYS = ["code_module", "code_presentation"]

vle_w = studentVle_weekly[studentVle_weekly["engagement_dim"].isin(dim_keep)].copy()

# attach course length + drop weeks beyond course length
vle_w = vle_w.merge(
    course_windows[COURSE_KEYS + ["course_weeks"]],
    on=COURSE_KEYS,
    how="inner",
    validate="m:1"
)
vle_w = vle_w[(vle_w["week"] >= 1) & (vle_w["week"] <= vle_w["course_weeks"])].copy()

# --- HARDEN TYPES BEFORE NORMALIZATION ---
for c in ["code_module", "code_presentation"]:
    vle_w[c] = vle_w[c].astype(str).str.strip()
    course_windows[c] = course_windows[c].astype(str).str.strip()

vle_w["engagement_dim"] = vle_w["engagement_dim"].astype(str).str.strip().str.lower()

vle_w["week"] = pd.to_numeric(vle_w["week"], errors="coerce")
vle_w = vle_w.dropna(subset=["week"]).copy()
vle_w["week"] = vle_w["week"].astype(int)

course_week_dim = (
    vle_w.groupby(COURSE_KEYS + ["week", "engagement_dim"], as_index=False)
         .agg(course_clicks=("sum_click", "sum"))
)

vle_w = vle_w.merge(
    course_week_dim,
    on=COURSE_KEYS + ["week", "engagement_dim"],
    how="left",
    validate="m:1"
)

print("course_clicks missing rate:", vle_w["course_clicks"].isna().mean())

vle_w["share_clicks"] = np.where(
    vle_w["course_clicks"] > 0,
    vle_w["sum_click"] / (vle_w["course_clicks"] + 1e-9),
    0.0
)

# ✅ 4) ADD THE CHECK HERE (right after share_clicks)
chk_vle = (
    vle_w.groupby(COURSE_KEYS + ["week","engagement_dim"], as_index=False)
         .agg(sum_share=("share_clicks","sum"),
              course_clicks=("course_clicks","first"))
)

print(chk_vle["sum_share"].describe())
print(((chk_vle["sum_share"] > 0.98) & (chk_vle["sum_share"] < 1.02)).mean())


weekly_share_wide = (
    vle_w.pivot_table(index=KEYS, columns=["week","engagement_dim"], values="share_clicks", fill_value=0.0)
         .reset_index()
)

# ✅ 1) Flatten the KEY columns ('id_student','') -> 'id_student'
weekly_share_wide.columns = [
    c[0] if (isinstance(c, tuple) and (c[1] == "" or c[1] is None)) else c
    for c in weekly_share_wide.columns
]


# def _col_to_name_norm(col):
#     if isinstance(col, str):
#         return col
#     if isinstance(col, tuple):
#         wk, dim = None, None
#         for p in col:
#             if isinstance(p, (int, np.integer, float, np.floating)) and wk is None:
#                 wk = int(p)
#             if isinstance(p, str):
#                 dim = p
#         if wk is not None and dim is not None:
#             return f"wk{wk:02d}_{dim}_norm"
#     return str(col)

# ✅ 2) Rename (week, dim) tuples into wkXX_dim_norm
def _col_to_name_norm(col):
    if isinstance(col, tuple):
        wk, dim = col
        return f"wk{int(wk):02d}_{dim}_norm"
    return col

weekly_share_wide = weekly_share_wide.rename(columns=_col_to_name_norm)


# weekly_share_wide.columns = [_col_to_name_norm(c) for c in weekly_share_wide.columns]

master_table = base_static.merge(weekly_share_wide, on=KEYS, how="left")
print("merged master_table shape:", master_table.shape)

norm_weekly_cols = [c for c in weekly_share_wide.columns if c not in KEYS]
master_table[norm_weekly_cols] = master_table[norm_weekly_cols].fillna(0.0)
# del weekly_dim_long, weekly_dim_wide, weekly_cols
# gc.collect()

course_clicks missing rate: 0.0
count    2.337000e+03
mean     1.000000e+00
std      7.822672e-11
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: sum_share, dtype: float64
1.0
merged master_table shape: (29915, 148)


In [14]:
master_table.head()

,code_module,code_presentation,id_student,num_of_prev_attempts,studied_credits,target_at_risk,gender_bin,disability_bin,age_band_ord,age_band_missing_flag,...,wk36_social_norm,wk37_assessment_norm,wk37_content_norm,wk37_social_norm,wk38_assessment_norm,wk38_content_norm,wk38_social_norm,wk39_assessment_norm,wk39_content_norm,wk39_social_norm
0,AAA,2013J,11391,0,240,0,1,0,2,0,...,0.003404,0.0,0.001333,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1,AAA,2013J,28400,0,60,0,0,0,1,0,...,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
2,AAA,2013J,30268,0,60,1,0,1,1,0,...,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
3,AAA,2013J,31604,0,60,0,0,0,1,0,...,0.008169,0.0,0.001333,0.0,0.0,0.003252,0.0,0.0,0.0,0.0
4,AAA,2013J,32885,0,60,0,0,0,0,0,...,0.002723,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


In [15]:
eps = 1e-9


# def add_master_weekly_features_courseaware(df: pd.DataFrame, course_weeks_col="course_weeks", eps=1e-9):
#     # detect max week present
#     week_pat = re.compile(r"^wk(\d{2})_(content|assessment|social)$")
#     weeks = sorted({int(week_pat.match(c).group(1)) for c in df.columns if week_pat.match(c)})
#     max_w = max(weeks) if weeks else 0
#     if max_w == 0:
#         return df.copy()

#     # build matrices (fill missing cols with 0)
#     content_cols = [f"wk{wk:02d}_content" for wk in weeks]
#     assess_cols  = [f"wk{wk:02d}_assessment" for wk in weeks]
#     social_cols  = [f"wk{wk:02d}_social" for wk in weeks]

#     Xc = df.reindex(columns=content_cols, fill_value=0.0).to_numpy(float)
#     Xa = df.reindex(columns=assess_cols,  fill_value=0.0).to_numpy(float)
#     Xs = df.reindex(columns=social_cols,  fill_value=0.0).to_numpy(float)
#     Xt = Xc + Xa + Xs  # weekly totals (all weeks)

#     # course length per row
#     cw = pd.to_numeric(df[course_weeks_col], errors="coerce").fillna(max_w).clip(lower=1, upper=max_w).to_numpy(int)

#     # mask weeks > course_weeks
#     week_index = np.arange(1, len(weeks) + 1)[None, :]          # shape (1, W)
#     valid_mask = (week_index <= cw[:, None]).astype(float)       # shape (N, W)

#     Xc_v = Xc * valid_mask
#     Xa_v = Xa * valid_mask
#     Xs_v = Xs * valid_mask
#     Xt_v = Xt * valid_mask

#     # aggregates
#     content_engagement    = Xc_v.sum(axis=1)
#     assessment_engagement = Xa_v.sum(axis=1)
#     social_engagement     = Xs_v.sum(axis=1)
#     total_engagement      = Xt_v.sum(axis=1)

#     content_ratio    = content_engagement / (total_engagement + eps)
#     assessment_ratio = assessment_engagement / (total_engagement + eps)
#     social_ratio     = social_engagement / (total_engagement + eps)

#     active_weeks = (Xt_v > 0).sum(axis=1).astype(int)
#     weeks_observed = cw
#     active_weeks_ratio = active_weeks / (weeks_observed + eps)

#     # entropy (only valid weeks)
#     row_sum = Xt_v.sum(axis=1, keepdims=True)
#     P = np.divide(Xt_v, row_sum + eps)

#     logP = np.zeros_like(P)
#     mask = P > 0
#     logP[mask] = np.log(P[mask])
#     engagement_entropy = -(P * logP).sum(axis=1)

#     # normalize by log(course_weeks) per row
#     engagement_entropy_norm = engagement_entropy / (np.log(weeks_observed) + eps)

#     new_feat_df = pd.DataFrame({
#         "content_engagement": content_engagement,
#         "assessment_engagement": assessment_engagement,
#         "social_engagement": social_engagement,
#         "total_engagement": total_engagement,
#         "content_ratio": content_ratio,
#         "assessment_ratio": assessment_ratio,
#         "social_ratio": social_ratio,
#         "active_weeks": active_weeks,
#         "weeks_observed": weeks_observed,
#         "active_weeks_ratio": active_weeks_ratio,
#         "engagement_entropy": engagement_entropy,
#         "engagement_entropy_norm": engagement_entropy_norm,
#     }, index=df.index)

#     return pd.concat([df.copy(), new_feat_df], axis=1).copy()


def add_weekly_features_courseaware_from_norm(
    df: pd.DataFrame,
    course_weeks_col: str = "course_weeks",
    suffix: str = "_norm",
    eps: float = 1e-9
) -> pd.DataFrame:
    """
    Course-length-aware engineered features from normalized weekly columns.

    Expects weekly columns like:
      wk01_content{suffix}, wk01_assessment{suffix}, wk01_social{suffix}, ...

    Produces (all with the same suffix):
      content_engagement{suffix}, assessment_engagement{suffix}, social_engagement{suffix}, total_engagement{suffix}
      content_ratio{suffix}, assessment_ratio{suffix}, social_ratio{suffix}
      active_weeks{suffix}, weeks_observed{suffix}, active_weeks_ratio{suffix}
      engagement_entropy{suffix}, engagement_entropy_norm{suffix}

    Course-length awareness:
      - weeks beyond each row's course_weeks are masked out (ignored)
      - entropy is normalized by log(course_weeks) per row
    """

    # 1) Find which weeks exist in the dataframe for this suffix
    week_pat = re.compile(rf"^wk(\d{{2}})_(content|assessment|social){re.escape(suffix)}$")
    weeks = sorted({int(week_pat.match(c).group(1)) for c in df.columns if week_pat.match(c)})
    if not weeks:
        return df.copy()

    max_w = max(weeks)

    # 2) Build matrices for the 3 dims (missing cols are treated as 0)
    content_cols = [f"wk{wk:02d}_content{suffix}" for wk in weeks]
    assess_cols  = [f"wk{wk:02d}_assessment{suffix}" for wk in weeks]
    social_cols  = [f"wk{wk:02d}_social{suffix}" for wk in weeks]

    Xc = df.reindex(columns=content_cols, fill_value=0.0).to_numpy(dtype=float)
    Xa = df.reindex(columns=assess_cols,  fill_value=0.0).to_numpy(dtype=float)
    Xs = df.reindex(columns=social_cols,  fill_value=0.0).to_numpy(dtype=float)
    Xt = Xc + Xa + Xs  # (N, W)

    # 3) Course length per row
    if course_weeks_col not in df.columns:
        raise KeyError(
            f"'{course_weeks_col}' not found. Available cols include: "
            + ", ".join(list(df.columns[:30])) + ("..." if df.shape[1] > 30 else "")
        )

    cw = pd.to_numeric(df[course_weeks_col], errors="coerce").fillna(max_w).clip(1, max_w).to_numpy(dtype=int)

    # 4) Mask weeks beyond course length (course-length aware)
    week_index = np.array(weeks, dtype=int)[None, :]       # actual week numbers, shape (1, W)
    valid_mask = (week_index <= cw[:, None]).astype(float) # shape (N, W)

    Xc_v = Xc * valid_mask
    Xa_v = Xa * valid_mask
    Xs_v = Xs * valid_mask
    Xt_v = Xt * valid_mask

    # 5) Aggregates
    content_eng = Xc_v.sum(axis=1)
    assess_eng  = Xa_v.sum(axis=1)
    social_eng  = Xs_v.sum(axis=1)
    total_eng   = Xt_v.sum(axis=1)

    content_ratio    = content_eng / (total_eng + eps)
    assessment_ratio = assess_eng  / (total_eng + eps)
    social_ratio     = social_eng  / (total_eng + eps)

    active_weeks = (Xt_v > 0).sum(axis=1).astype(int)
    weeks_observed = cw
    active_weeks_ratio = active_weeks / (weeks_observed + eps)

    # 6) Entropy over weeks (vectorized, course-length aware)
    row_sum = Xt_v.sum(axis=1, keepdims=True)
    P = np.divide(Xt_v, row_sum + eps)  # safe divide

    logP = np.zeros_like(P)
    m = P > 0
    logP[m] = np.log(P[m])

    ent = -(P * logP).sum(axis=1)
    ent_norm = ent / (np.log(weeks_observed) + eps)

    # 7) Append engineered columns in one concat (avoids fragmentation)
    new_feat = pd.DataFrame({
        f"content_engagement{suffix}": content_eng,
        f"assessment_engagement{suffix}": assess_eng,
        f"social_engagement{suffix}": social_eng,
        f"total_engagement{suffix}": total_eng,
        f"content_ratio{suffix}": content_ratio,
        f"assessment_ratio{suffix}": assessment_ratio,
        f"social_ratio{suffix}": social_ratio,
        f"active_weeks{suffix}": active_weeks,
        f"weeks_observed{suffix}": weeks_observed,
        f"active_weeks_ratio{suffix}": active_weeks_ratio,
        f"engagement_entropy{suffix}": ent,
        f"engagement_week_entropy_norm{suffix}": ent_norm,
    }, index=df.index)

    out = pd.concat([df.copy(), new_feat], axis=1)
    return out.copy()

master_table = add_weekly_features_courseaware_from_norm(
    master_table,
    course_weeks_col="course_weeks",
    suffix="_norm",
    eps=eps
)

In [16]:
print(master_table.columns.tolist())

['code_module', 'code_presentation', 'id_student', 'num_of_prev_attempts', 'studied_credits', 'target_at_risk', 'gender_bin', 'disability_bin', 'age_band_ord', 'age_band_missing_flag', 'highest_education_ord', 'highest_education_missing_flag', 'nation_England', 'nation_Northern_Ireland', 'nation_Scotland', 'nation_Wales', 'imd_5cat', 'imd_ordinal', 'date_registration', 'date_unregistration', 'registration_missing_flag', 'unregistered_flag', 'unregistration_week', 'late_registration_flag', 'module_presentation_length', 'course_type_stem', 'course_weeks', 'q1_end', 'q2_end', 'q3_end', 'q4_end', 'wk01_assessment_norm', 'wk01_content_norm', 'wk01_social_norm', 'wk02_assessment_norm', 'wk02_content_norm', 'wk02_social_norm', 'wk03_assessment_norm', 'wk03_content_norm', 'wk03_social_norm', 'wk04_assessment_norm', 'wk04_content_norm', 'wk04_social_norm', 'wk05_assessment_norm', 'wk05_content_norm', 'wk05_social_norm', 'wk06_assessment_norm', 'wk06_content_norm', 'wk06_social_norm', 'wk07_asse

In [17]:
# Attach course windows to weekly VLE
vle_q = studentVle_weekly.merge(
    course_windows[COURSE_KEYS + ["course_weeks", "q1_end", "q2_end", "q3_end", "q4_end"]],
    on=COURSE_KEYS, how="inner", validate="m:1"
).copy()

vle_q = vle_q[(vle_q["week"] >= 1) & (vle_q["week"] <= vle_q["course_weeks"])].copy()

def build_quarter_norm_features(vle_df: pd.DataFrame, q_end_col: str, q_label: str) -> pd.DataFrame:
    df = vle_df[vle_df["week"] <= vle_df[q_end_col]].copy()
    dim_keep = {"content", "assessment", "social"}

    student_dim = (
        df[df["engagement_dim"].isin(dim_keep)]
        .groupby(KEYS + ["engagement_dim"], as_index=False)
        .agg(clicks=("sum_click", "sum"))
    )

    course_dim = (
        df[df["engagement_dim"].isin(dim_keep)]
        .groupby(COURSE_KEYS + ["engagement_dim"], as_index=False)
        .agg(course_clicks=("sum_click", "sum"))
    )

    student_dim = student_dim.merge(course_dim, on=COURSE_KEYS + ["engagement_dim"], how="left", validate="m:1")
    student_dim["norm"] = np.where(student_dim["course_clicks"] > 0,
                                   student_dim["clicks"] / student_dim["course_clicks"], 0.0)

    dim_wide = (
        student_dim.pivot_table(index=KEYS, columns="engagement_dim", values="norm", fill_value=0.0)
        .reset_index()
        .rename(columns={
            "content": f"norm_content_{q_label}",
            "assessment": f"norm_assessment_{q_label}",
            "social": f"norm_social_{q_label}",
        })
    )

    student_total = df.groupby(KEYS, as_index=False).agg(total_clicks=("sum_click", "sum"))
    course_total  = df.groupby(COURSE_KEYS, as_index=False).agg(course_total_clicks=("sum_click", "sum"))

    total_norm = student_total.merge(course_total, on=COURSE_KEYS, how="left", validate="m:1")
    total_norm[f"norm_total_{q_label}"] = np.where(total_norm["course_total_clicks"] > 0,
                                                   total_norm["total_clicks"] / total_norm["course_total_clicks"], 0.0)
    total_norm = total_norm[KEYS + [f"norm_total_{q_label}"]]

    out = total_norm.merge(dim_wide, on=KEYS, how="left")
    for c in [f"norm_content_{q_label}", f"norm_assessment_{q_label}", f"norm_social_{q_label}"]:
        if c not in out.columns:
            out[c] = 0.0
        else:
            out[c] = out[c].fillna(0.0)

    return out

q1_norm = build_quarter_norm_features(vle_q, "q1_end", "Q1")
q2_norm = build_quarter_norm_features(vle_q, "q2_end", "Q2")
q3_norm = build_quarter_norm_features(vle_q, "q3_end", "Q3")
q4_norm = build_quarter_norm_features(vle_q, "q4_end", "Q4")


In [18]:
# =========================
# 10) AP score: full + per quarter
# =========================
assess_clean = assessments.copy()
assess_clean["assessment_type"] = assess_clean["assessment_type"].astype(str).str.lower()
assess_clean = assess_clean[assess_clean["assessment_type"] != "exam"].copy()
assess_clean["date"] = pd.to_numeric(assess_clean["date"], errors="coerce")
assess_clean = assess_clean.dropna(subset=["date"])
assess_clean = assess_clean[["id_assessment", "code_module", "code_presentation", "date"]].rename(columns={"date": "due_date"})

sa = studentAssessment.copy()
sa = sa[sa["date_submitted"].notna()].copy()
sa["date_submitted"] = pd.to_numeric(sa["date_submitted"], errors="coerce").clip(lower=0)
sa = sa[["id_assessment", "id_student", "date_submitted"]]

sub = sa.merge(assess_clean, on="id_assessment", how="inner", validate="m:1")
sub = sub.merge(course_windows[COURSE_KEYS + ["q1_end", "q2_end", "q3_end", "q4_end"]], on=COURSE_KEYS, how="inner", validate="m:1")

# FULL Smin per assessment (full-course)
smin_full = (
    sub.groupby(COURSE_KEYS + ["id_assessment"], as_index=False)
       .agg(Smin=("date_submitted", "min"))
)
sub_full = sub.merge(smin_full, on=COURSE_KEYS + ["id_assessment"], how="left", validate="m:1")
sub_full["Wa"] = sub_full["due_date"] - sub_full["Smin"]
sub_full["delta"] = sub_full["due_date"] - sub_full["date_submitted"]
sub_full["Ts"] = np.where(sub_full["Wa"] > 0, sub_full["delta"] / (sub_full["Wa"] + eps), 0.0)
sub_full["late_flag"] = (sub_full["date_submitted"] > sub_full["due_date"]).astype(int)

ap_full = (
    sub_full.groupby(KEYS, as_index=False)
            .agg(AP_full=("Ts", "mean"),
                 n_submissions=("Ts", "count"),
                 num_late_submissions=("late_flag", "sum"))
)

master_table = master_table.merge(ap_full, on=KEYS, how="left")
master_table["AP_full_missing"] = master_table["AP_full"].isna().astype(int)
master_table["AP_full"] = master_table["AP_full"].fillna(0.0)
master_table["n_submissions"] = master_table["n_submissions"].fillna(0).astype(int)
master_table["num_late_submissions"] = master_table["num_late_submissions"].fillna(0).astype(int)

In [19]:
def compute_ap_quarter(sub_df: pd.DataFrame, q_end_col: str, q_label: str) -> pd.DataFrame:
    dfq = sub_df[sub_df["date_submitted"] <= sub_df[q_end_col]].copy()

    smin_q = (
        dfq.groupby(COURSE_KEYS + ["id_assessment"], as_index=False)
           .agg(Smin_q=("date_submitted", "min"))
    )
    dfq = dfq.merge(smin_q, on=COURSE_KEYS + ["id_assessment"], how="left", validate="m:1")

    dfq["Wa_q"] = dfq["due_date"] - dfq["Smin_q"]
    dfq["delta_q"] = dfq["due_date"] - dfq["date_submitted"]
    dfq["Ts_q"] = np.where(dfq["Wa_q"] > 0, dfq["delta_q"] / (dfq["Wa_q"] + eps), 0.0)
    dfq["late_q"] = (dfq["date_submitted"] > dfq["due_date"]).astype(int)

    out = (
        dfq.groupby(KEYS, as_index=False)
           .agg(**{
               f"AP_{q_label}": ("Ts_q", "mean"),
               f"n_submissions_{q_label}": ("Ts_q", "count"),
               f"num_late_submissions_{q_label}": ("late_q", "sum"),
           })
    )
    return out

ap_Q1 = compute_ap_quarter(sub, "q1_end", "Q1")
ap_Q2 = compute_ap_quarter(sub, "q2_end", "Q2")
ap_Q3 = compute_ap_quarter(sub, "q3_end", "Q3")
ap_Q4 = compute_ap_quarter(sub, "q4_end", "Q4")

In [20]:
# =========================
# 11) Quarter datasets (Q1..Q4) + quarter engineered features
# =========================
def add_quarter_features(df: pd.DataFrame, q_label: str, eps: float = 1e-9) -> pd.DataFrame:
    t = f"norm_total_{q_label}"
    c = f"norm_content_{q_label}"
    a = f"norm_assessment_{q_label}"
    s = f"norm_social_{q_label}"

    for col in [t, c, a, s]:
        if col not in df.columns:
            df[col] = 0.0
        df[col] = df[col].fillna(0.0)

    df[f"content_ratio_{q_label}"]    = df[c] / (df[t] + eps)
    df[f"assessment_ratio_{q_label}"] = df[a] / (df[t] + eps)
    df[f"social_ratio_{q_label}"]     = df[s] / (df[t] + eps)

    vals = df[[c, a, s]].to_numpy(float)
    row_sum = vals.sum(axis=1, keepdims=True)
    P = vals / (row_sum + eps)
    logP = np.zeros_like(P)
    mask = P > 0
    logP[mask] = np.log(P[mask])
    dim_entropy = -(P * logP).sum(axis=1)
    df[f"dim_entropy_{q_label}"] = dim_entropy
    df[f"dim_entropy_norm_{q_label}"] = dim_entropy / (np.log(3) + eps)

    ap = f"AP_{q_label}"
    if ap in df.columns:
        df[f"{ap}_missing"] = df[ap].isna().astype(int)
        df[ap] = df[ap].fillna(0.0)

    return df

quarter_Q1 = add_quarter_features(base_static.merge(q1_norm, on=KEYS, how="left").merge(ap_Q1, on=KEYS, how="left"), "Q1", eps)
quarter_Q2 = add_quarter_features(base_static.merge(q2_norm, on=KEYS, how="left").merge(ap_Q2, on=KEYS, how="left"), "Q2", eps)
quarter_Q3 = add_quarter_features(base_static.merge(q3_norm, on=KEYS, how="left").merge(ap_Q3, on=KEYS, how="left"), "Q3", eps)
quarter_Q4 = add_quarter_features(base_static.merge(q4_norm, on=KEYS, how="left").merge(ap_Q4, on=KEYS, how="left"), "Q4", eps)

# Optional cleanup
del vle_q, q1_norm, q2_norm, q3_norm, q4_norm, ap_Q1, ap_Q2, ap_Q3, ap_Q4, sub_full
gc.collect()

0

In [21]:
print("base_static:", base_static.shape)
print("master_table:", master_table.shape)
print("quarter_Q1:", quarter_Q1.shape, "quarter_Q2:", quarter_Q2.shape, "quarter_Q3:", quarter_Q3.shape, "quarter_Q4:", quarter_Q4.shape)

base_static: (29915, 31)
master_table: (29915, 164)
quarter_Q1: (29915, 44) quarter_Q2: (29915, 44) quarter_Q3: (29915, 44) quarter_Q4: (29915, 44)


In [22]:
list(master_table.columns)

['code_module',
 'code_presentation',
 'id_student',
 'num_of_prev_attempts',
 'studied_credits',
 'target_at_risk',
 'gender_bin',
 'disability_bin',
 'age_band_ord',
 'age_band_missing_flag',
 'highest_education_ord',
 'highest_education_missing_flag',
 'nation_England',
 'nation_Northern_Ireland',
 'nation_Scotland',
 'nation_Wales',
 'imd_5cat',
 'imd_ordinal',
 'date_registration',
 'date_unregistration',
 'registration_missing_flag',
 'unregistered_flag',
 'unregistration_week',
 'late_registration_flag',
 'module_presentation_length',
 'course_type_stem',
 'course_weeks',
 'q1_end',
 'q2_end',
 'q3_end',
 'q4_end',
 'wk01_assessment_norm',
 'wk01_content_norm',
 'wk01_social_norm',
 'wk02_assessment_norm',
 'wk02_content_norm',
 'wk02_social_norm',
 'wk03_assessment_norm',
 'wk03_content_norm',
 'wk03_social_norm',
 'wk04_assessment_norm',
 'wk04_content_norm',
 'wk04_social_norm',
 'wk05_assessment_norm',
 'wk05_content_norm',
 'wk05_social_norm',
 'wk06_assessment_norm',
 'wk0

In [23]:
list(quarter_Q1.columns)

['code_module',
 'code_presentation',
 'id_student',
 'num_of_prev_attempts',
 'studied_credits',
 'target_at_risk',
 'gender_bin',
 'disability_bin',
 'age_band_ord',
 'age_band_missing_flag',
 'highest_education_ord',
 'highest_education_missing_flag',
 'nation_England',
 'nation_Northern_Ireland',
 'nation_Scotland',
 'nation_Wales',
 'imd_5cat',
 'imd_ordinal',
 'date_registration',
 'date_unregistration',
 'registration_missing_flag',
 'unregistered_flag',
 'unregistration_week',
 'late_registration_flag',
 'module_presentation_length',
 'course_type_stem',
 'course_weeks',
 'q1_end',
 'q2_end',
 'q3_end',
 'q4_end',
 'norm_total_Q1',
 'norm_assessment_Q1',
 'norm_content_Q1',
 'norm_social_Q1',
 'AP_Q1',
 'n_submissions_Q1',
 'num_late_submissions_Q1',
 'content_ratio_Q1',
 'assessment_ratio_Q1',
 'social_ratio_Q1',
 'dim_entropy_Q1',
 'dim_entropy_norm_Q1',
 'AP_Q1_missing']

In [ ]:
col = "wk05_content_norm"
chk = master_table.groupby(COURSE_KEYS)[col].sum()

chk.describe()
chk.head()

# how many are close to 1?
((chk > 0.98) & (chk < 1.02)).mean()
# how many are exactly 0?
(chk == 0).mean()

In [ ]:
vle_w["week"].dtype, vle_w["week"].isna().mean(), vle_w["week"].min(), vle_w["week"].max()
vle_w["week"].value_counts().head(10)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=915c48ff-4287-4f39-af89-419738985e00' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>